In [ ]:
#!/usr/bin/env python3
"""
SuperAnalytics - Versión Final (GUI + Filtros Temporales)
=========================================================
1. Abre ventana emergente (forzada al frente) para elegir archivo.
2. Lee Compras y Ventas.
3. Genera hoja "Base_Detallada" con Semana/Mes/Año para filtros.
4. Genera hoja "Resumen" con métricas totales.
"""

import pandas as pd
import os
import tkinter as tk
from tkinter import filedialog, messagebox
import datetime

# Diccionario para que los meses salgan con nombre en español
MESES_ESP = {
    1: "01-Enero", 2: "02-Febrero", 3: "03-Marzo", 4: "04-Abril",
    5: "05-Mayo", 6: "06-Junio", 7: "07-Julio", 8: "08-Agosto",
    9: "09-Septiembre", 10: "10-Octubre", 11: "11-Noviembre", 12: "12-Diciembre"
}

def leer_hoja(path, sheet_index, nombre_hoja, tipo_movimiento):
    """Lee una hoja, normaliza columnas y prepara fechas."""
    try:
        df = pd.read_excel(path, sheet_name=sheet_index)
    except Exception as e:
        print(f"❌ Error leyendo hoja {sheet_index}: {e}")
        return pd.DataFrame()

    print(f"  📄 {nombre_hoja}: {len(df):,} filas")

    # Normalización de nombres de columnas
    col_map = {}
    for c in df.columns:
        cl = str(c).strip().lower()
        if cl in ("f.ingreso", "f. ingreso", "fecha", "fingreso", "date", "dia"):
            col_map[c] = "fecha"
        elif cl in ("descripción artículo", "articulo", "artículo", "producto", "item"):
            col_map[c] = "articulo"
        elif cl in ("cantidad", "cant", "qty", "unidades"):
            col_map[c] = "cantidad"
        elif cl.strip() in ("precio", "costo", "precio unitario", "unitario"):
            col_map[c] = "precio"
        elif cl.strip() in ("monto", "total", "importe", "valor"):
            col_map[c] = "monto"
        elif cl in ("descripción", "descripcion", "categoria", "categoría", "rubro"):
            col_map[c] = "categoria"

    df = df.rename(columns=col_map)

    # Asegurar columnas mínimas
    for col in ["articulo", "cantidad", "precio", "monto", "categoria", "fecha"]:
        if col not in df.columns:
            if col in ("cantidad", "precio", "monto"):
                df[col] = 0
            else:
                df[col] = ""

    df["articulo"] = df["articulo"].astype(str).str.strip()
    df = df[df["articulo"] != ""].copy()
    
    # ─── LOGICA DE FECHAS (Semana/Mes/Año) ───
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    
    # Extraer datos temporales
    df["Año"] = df["fecha"].dt.year.fillna(0).astype(int)
    df["Semana"] = df["fecha"].dt.isocalendar().week.fillna(0).astype(int)
    df["Mes_Num"] = df["fecha"].dt.month.fillna(0).astype(int)
    df["Mes"] = df["Mes_Num"].map(MESES_ESP).fillna("Sin Fecha")

    # Limpieza numérica
    for col in ["cantidad", "precio", "monto"]:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(r"[^\d.,\-]", "", regex=True).str.replace(",", "."), 
            errors="coerce"
        ).fillna(0)

    df["categoria"] = df["categoria"].astype(str).str.strip().replace({"": "General", "nan": "General"})

    # Calcular monto si falta
    mask = df["monto"] == 0
    df.loc[mask, "monto"] = df.loc[mask, "cantidad"] * df.loc[mask, "precio"]

    # Etiquetar (Compra o Venta) para la base unificada
    df["Tipo"] = tipo_movimiento

    return df


def agrupar(df):
    """Calcula totales por artículo para el resumen."""
    if df.empty:
        return pd.DataFrame()

    grouped = df.groupby("articulo").agg(
        categoria=("categoria", "first"),
        cantidad_total=("cantidad", "sum"),
        monto_total=("monto", "sum"),
        precio_promedio=("precio", "mean"),
        precio_min=("precio", "min"),
        precio_max=("precio", "max"),
        cantidad_registros=("articulo", "count"),
    ).reset_index()

    return grouped


def main():
    # ─── CONFIGURACIÓN DE VENTANA (SOLUCIÓN AL PROBLEMA VISUAL) ───
    root = tk.Tk()
    root.withdraw() # Oculta la ventana base fea
    root.attributes('-topmost', True) # ⚡ FUERZA LA VENTANA SIEMPRE ENCIMA ⚡
    
    print("⏳ Abriendo ventana para seleccionar archivo...")
    
    # 1. Seleccionar Origen
    input_path = filedialog.askopenfilename(
        parent=root,
        title="Seleccionar archivo de origen (Excel)",
        filetypes=[("Archivos Excel", "*.xlsx *.xls")]
    )
    
    # Quitamos el 'siempre encima' momentáneamente para que no moleste
    root.attributes('-topmost', False)

    if not input_path:
        print("❌ Cancelado por el usuario.")
        root.destroy()
        return

    # Preparar ruta destino
    base_dir = os.path.dirname(input_path)
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    default_output = f"{base_name}_PROCESADO.xlsx"

    print("⏳ Seleccionando dónde guardar...")
    root.attributes('-topmost', True) # ⚡ Forzamos otra vez para guardar
    
    # 2. Seleccionar Destino
    output_path = filedialog.asksaveasfilename(
        parent=root,
        title="Guardar archivo procesado como...",
        initialdir=base_dir,
        initialfile=default_output,
        defaultextension=".xlsx",
        filetypes=[("Archivos Excel", "*.xlsx")]
    )
    
    root.destroy() # Ya no necesitamos la interfaz gráfica

    if not output_path:
        return

    try:
        print(f"\n🚀 Procesando datos de: {os.path.basename(input_path)}")
        
        # Leer Hojas (Asumimos Hoja 1 = Compras, Hoja 2 = Ventas)
        df_compras = leer_hoja(input_path, 0, "Compras", "COMPRA")
        df_ventas = leer_hoja(input_path, 1, "Ventas", "VENTA")

        # ─── PASO 1: BASE DETALLADA (Para tus filtros de fecha) ───
        # Unimos todo en una lista larga
        cols_detalle = ["fecha", "Año", "Mes", "Semana", "Tipo", "articulo", "categoria", "cantidad", "precio", "monto"]
        
        # Concatenar si existen columnas
        df_master = pd.concat([
            df_compras[cols_detalle] if not df_compras.empty else pd.DataFrame(),
            df_ventas[cols_detalle] if not df_ventas.empty else pd.DataFrame()
        ], ignore_index=True)
        
        if not df_master.empty:
            df_master = df_master.sort_values("fecha", ascending=False)

        # ─── PASO 2: RESUMEN POR ARTÍCULO (Totales) ───
        print("⚙️  Calculando resumen...")
        compras_agr = agrupar(df_compras)
        ventas_agr = agrupar(df_ventas)

        # Renombrar columnas para merge
        if not compras_agr.empty:
            compras_agr.columns = ["articulo", "cat_c", "cant_comprada", "monto_compra", "costo_prom", "costo_min", "costo_max", "reg_compra"]
        if not ventas_agr.empty:
            ventas_agr.columns = ["articulo", "cat_v", "cant_vendida", "monto_venta", "precio_prom", "precio_min", "precio_max", "reg_venta"]

        # Unir Compras y Ventas
        if compras_agr.empty and ventas_agr.empty:
             messagebox.showerror("Error", "No se encontraron datos.")
             return
        elif compras_agr.empty:
             resumen = ventas_agr.copy()
             resumen["categoria"] = resumen["cat_v"]
             for c in ["cant_comprada", "monto_compra", "costo_prom"]: resumen[c] = 0
        elif ventas_agr.empty:
             resumen = compras_agr.copy()
             resumen["categoria"] = resumen["cat_c"]
             for c in ["cant_vendida", "monto_venta", "precio_prom"]: resumen[c] = 0
        else:
            resumen = pd.merge(compras_agr, ventas_agr, on="articulo", how="outer")
            resumen["categoria"] = resumen["cat_c"].fillna(resumen["cat_v"]).fillna("General")

        resumen = resumen.fillna(0)
        
        # KPIs Financieros
        resumen["ganancia"] = resumen["monto_venta"] - resumen["monto_compra"]
        resumen["margen_%"] = resumen.apply(
            lambda r: round(((r["precio_prom"] - r["costo_prom"]) / r["precio_prom"]) * 100, 1) if r["precio_prom"] > 0 else 0, axis=1
        )
        
        # Estado
        def calc_estado(r):
            if r["cant_comprada"] > 0 and r["cant_vendida"] == 0: return "⚠️ Stock Inmóvil"
            if r["cant_comprada"] == 0 and r["cant_vendida"] > 0: return "🔴 Sin Reposición"
            if r["ganancia"] < 0: return "📉 Pérdida"
            return "✅ OK"
        resumen["estado"] = resumen.apply(calc_estado, axis=1)

        # Seleccionar columnas finales limpias
        cols_final = ["articulo", "categoria", "estado", "margen_%", "ganancia", 
                      "cant_comprada", "cant_vendida", "monto_compra", "monto_venta", 
                      "costo_prom", "precio_prom"]
        resumen = resumen[[c for c in cols_final if c in resumen.columns]]

        # ─── GUARDAR EXCEL ───
        print(f"💾 Guardando archivo...")
        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
            df_master.to_excel(writer, sheet_name="Base_Detallada", index=False)
            resumen.to_excel(writer, sheet_name="Resumen_Articulos", index=False)
            
            # Top ventas
            if not resumen.empty:
                top = resumen.sort_values("monto_venta", ascending=False).head(20)
                top.to_excel(writer, sheet_name="Top_20_Ventas", index=False)

        # ─── FORMATO FINAL (Colores y ancho de columnas) ───
        from openpyxl import load_workbook
        from openpyxl.styles import Font, PatternFill
        
        wb = load_workbook(output_path)
        header_fill = PatternFill("solid", fgColor="2c3e50")
        header_font = Font(bold=True, color="FFFFFF")
        
        for ws in wb.worksheets:
            # Poner bonito el encabezado
            for cell in ws[1]:
                cell.fill = header_fill
                cell.font = header_font
            # Activar filtros automáticos de Excel
            ws.auto_filter.ref = ws.dimensions
            # Ajustar ancho columnas
            for col in range(1, ws.max_column + 1):
                ws.column_dimensions[ws.cell(1, col).column_letter].width = 15

        wb.save(output_path)

        # Mensaje final
        print("✅ ¡PROCESO TERMINADO!")
        messagebox.showinfo("Éxito", f"Archivo generado:\n{os.path.basename(output_path)}")

    except Exception as e:
        import traceback
        traceback.print_exc()
        messagebox.showerror("Error", f"Ocurrió un error:\n{str(e)}")

if __name__ == "__main__":
    main()

⏳ Abriendo ventana para seleccionar archivo...
⏳ Seleccionando dónde guardar...

🚀 Procesando datos de: ejemplo_supermercado_1.xlsx
  📄 Compras: 16,240 filas
  📄 Ventas: 61,351 filas
⚙️  Calculando resumen...
💾 Guardando archivo...
✅ ¡PROCESO TERMINADO!
